# Simulación gas ideal

## Configuración necesaria

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
from matplotlib.animation import FuncAnimation

# Configurar para animaciones interactivas
%matplotlib widget


## Simular un disco dentro de una caja y rebotando en sus paredes

Datos del sistema:
- $l_c=1$ (lado de la caja)
- $m_0= 1kg$ (masa del disco)

Condiciones inciales:
- $v_0=1$ (velocidad incial)
- dirección inicial de $\vec{v}$ aleatoria (np.random)

Datos simulación:
- $\Delta t_{sim} = 0,01$ (intervalo temporal entre pasos de simulación)
- nº de pasos de simulación = 10000
- nº de pasos entre representaciones = 10
- $\Delta t_{rep} = 0,01$ (ibtervalo de representación)


## Variables

In [ ]:
#==============================================================
# VARIABLES DE LA SIMULACIÓN
#==============================================================
l_c = 1 # Lado de la caja 
N = 10 # Numero de particulas
m_0 = 1 # Masa de las particulas
n_pasos_sim = 100000 # Numero de pasos de la simulacion
n_pasos_repres = 10 # Numero de pasos entre representaciones
v_0 = 1 # Velocidad inicial de las particulas, (modulo)
delta_t = 0.01 # Paso temporal de la simulacion
pausa_repres = 0.01 # Pausa entre representaciones (en segundos)
radio_particula = 0.0125 # Radio de las particulas

## Generamos la clase

In [ ]:

class dinamica_particulas_confinada_2D:
    """
    
    """
    def __init__(self, N_particulas, radio_particula, l_caja, v_0 = 1.0,m = 1.0):
        """
        
        """
        self.N = N_particulas
        self.r_radio = radio_particula
        self.l_c = l_caja
        self.v_0 = v_0
        self.m = m
        def inicializar_variables_necesarias():
            """
            Inicializa las variables necesarias para la simulacion:
            - Posiciones iniciales de las particulas (array de Nx2)
            - Velocidades iniciales de las particulas (array de Nx2)
            Argumentos:
            N = numero de particulas
            v_0 = velocidad inicial de las particulas (modulo)
            l_c = lado de la caja
            Retorna:
            variables_iniciales: dict con las variables iniciales
            """
            Particulas = {}
            for i in range(self.N):
                Particulas[f"particula_{i}"] = {
                    'posicion': np.zeros(2),
                    'velocidad': np.zeros(2)
                }
            # Inicializamos las posiciones
            posiciones = np.random.rand(self.N, 2) * self.l_c
            # Inicializamos las velocidades
            angulos = np.random.rand(self.N) * 2 * np.pi
            velocidades = np.zeros((self.N, 2))
            velocidades[:, 0] = self.v_0 * np.cos(angulos)
            velocidades[:, 1] = self.v_0 * np.sin(angulos)
            # Creamos el diccionario de variables iniciales
            for i in range(self.N):
                Particulas[f"particula_{i}"]['posicion'] = posiciones[i]
                Particulas[f"particula_{i}"]['velocidad'] = velocidades[i]
            return Particulas
        
        variables_iniciales = inicializar_variables_necesarias()
        self.situacion_inicial = variables_iniciales

    def calcular_distancias_particulas(self, Posiciones_totales, paso):
        """ Retorna matriz NxN de distancias """
        n_part = self.N
        distancias = np.zeros((n_part, n_part))

        for i in range(n_part):
            for j in range(i+1, n_part): # Solo calculamos la mitad superior
                d = np.linalg.norm(Posiciones_totales[f"particula_{i}"][f"paso_{paso-1}"] - Posiciones_totales[f"particula_{j}"][f"paso_{paso-1}"])
                distancias[i, j] = d
                distancias[j, i] = d # La matriz es simétrica
        return distancias
    
    def simular_dinamica(self, n_pasos, delta_t,choques):
        """
        
        """
        if choques == 'no':
            Posiciones_totales = {}
            velocidades = {}
            for i in range(self.N):
                Posiciones_totales[f"particula_{i}"] = {}
                velocidades[f"particula_{i}"] = self.situacion_inicial[f"particula_{i}"]['velocidad']
                for paso in range(n_pasos):
                    if paso == 0:
                        Posiciones_totales[f"particula_{i}"][f"paso_{paso}"] = self.situacion_inicial[f"particula_{i}"]['posicion']
                    else:
                        Posiciones_totales[f"particula_{i}"][f"paso_{paso}"] = Posiciones_totales[f"particula_{i}"][f"paso_{paso-1}"] + velocidades[f"particula_{i}"] * delta_t
                        # Verificar colisiones con paredes y ajustar velocidad
                        for dim in range(2): # 0 = x, 1 = y
                                if Posiciones_totales[f"particula_{i}"][f"paso_{paso}"][dim] <= 0 + 2*self.r_radio:
                                    velocidades[f"particula_{i}"][dim] *= -1
                                elif Posiciones_totales[f"particula_{i}"][f"paso_{paso}"][dim] >= self.l_c - 2*self.r_radio:
                                    velocidades[f"particula_{i}"][dim] *= -1
            self.dinamica_sin_choques = Posiciones_totales
            self.velocidades_finales = velocidades
            return Posiciones_totales
        else:
            Posiciones_totales = {}
            velocidades = {}
            for paso in range(n_pasos):
                rango = f"paso_{paso}"
                if paso != 0:
                    distancias = self.calcular_distancias_particulas(Posiciones_totales, paso)
                        
                    umbral = self.r_radio * 2 # Diámetro
                    for j in range(self.N):
                        for k in range(j + 1, self.N): # Iteramos pares únicos (j, k)
                            if distancias[j, k] <= umbral:
                                vector_ij = Posiciones_totales[f"particula_{k}"][f"paso_{paso-1}"] - Posiciones_totales[f"particula_{j}"][f"paso_{paso-1}"]
                                distancia_ij = np.linalg.norm(vector_ij)
                                vector_unitario = vector_ij / distancia_ij

                                velocidad_tangencial_j = np.dot(velocidades[f"particula_{j}"], vector_unitario) * vector_unitario
                                velocidad_tangencial_k = np.dot(velocidades[f"particula_{k}"], vector_unitario) * vector_unitario
                                velocidad_normal_j = velocidades[f"particula_{j}"] - velocidad_tangencial_j
                                velocidad_normal_k = velocidades[f"particula_{k}"] - velocidad_tangencial_k

                                velocidades[f"particula_{j}"] = velocidad_tangencial_k + velocidad_normal_j 
                                velocidades[f"particula_{k}"] = velocidad_tangencial_j + velocidad_normal_k
                            else:
                                continue
                
                for i in range(self.N):
                    if paso == 0:
                        Posiciones_totales[f"particula_{i}"] = {}
                        velocidades[f"particula_{i}"] = self.situacion_inicial[f"particula_{i}"]['velocidad']
                        Posiciones_totales[f"particula_{i}"][f"paso_0"] = self.situacion_inicial[f"particula_{i}"]['posicion']
                    else:
                        

                        Posiciones_totales[f"particula_{i}"][f"paso_{paso}"] = Posiciones_totales[f"particula_{i}"][f"paso_{paso-1}"] + velocidades[f"particula_{i}"] * delta_t
                        # Verificar colisiones con paredes y ajustar velocidad
                        for dim in range(2): # 0 = x, 1 = y
                            if Posiciones_totales[f"particula_{i}"][f"paso_{paso}"][dim] <= 0 + 2*self.r_radio:
                                velocidades[f"particula_{i}"][dim] *= -1
                            elif Posiciones_totales[f"particula_{i}"][f"paso_{paso}"][dim] >= self.l_c - 2*self.r_radio:
                                velocidades[f"particula_{i}"][dim] *= -1
            self.dinamica_con_choques = Posiciones_totales
            self.velocidades_finales = velocidades
            return Posiciones_totales

    def generar_animacion(self, distancia_frame, cant_frames,tiempo_frame, chocan = 'no'):
        """
        Docstring for generar_animacion
        
        :param self: Description
        """
        fig, ax = plt.subplots(figsize=(10,10))
        ax.set_xlim(0, self.l_c)
        ax.set_ylim(0, self.l_c)
        ax.set_aspect('equal')
        
        Particulas = []
        for i in range(self.N):
            particula, =ax.plot([], [], 'o', markersize=8)
            Particulas.append(particula)

        def init():
            for particula in Particulas:
                particula.set_data([], [])
            return Particulas

        if chocan == 'no':
            def animate(frame):
                paso = frame * distancia_frame
                for i, particula in enumerate(Particulas):
                    particula.set_data([self.dinamica_sin_choques[f"particula_{i}"][f"paso_{paso}"][0]], [self.dinamica_sin_choques[f"particula_{i}"][f"paso_{paso}"][1]])
                return Particulas
        else:
            def animate(frame):
                paso = frame * distancia_frame
                for i, particula in enumerate(Particulas):
                    particula.set_data([self.dinamica_con_choques[f"particula_{i}"][f"paso_{paso}"][0]], [self.dinamica_con_choques[f"particula_{i}"][f"paso_{paso}"][1]])
                return Particulas
            
        self.animacion = FuncAnimation(fig, animate, init_func=init, frames=cant_frames, 
                    blit=True, interval=tiempo_frame*1000, repeat=True)
        plt.show()
    def histograma_energia(self,choques = 'si'):
        """
        Docstring for histograma_energia
        :param self: Description
        """
        Energias = {}
        for particulas in range(self.N):
            velocidad_final = self.velocidades_finales[f"particula_{particulas}"]
            energia_cinetica = 0.5 * self.m * (np.linalg.norm(velocidad_final))**2
            Energias[f"particula_{particulas}"] = energia_cinetica
        energias_array = np.array(list(Energias.values()))
        plt.figure(figsize=(8,6))
        plt.hist(energias_array, bins=25, edgecolor='black',range=(0, 1))
        plt.title('Histograma de Energías Cinéticas Finales')
        plt.xlabel('Energía Cinética')
        plt.ylabel('Número de Partículas')
        plt.show()
        if choques == 'si':
            Energia_total_final = np.sum(energias_array)

            velocidad_inicial = np.zeros((self.N,2))
            Energias_iniciales = np.zeros(self.N)
            for particulas in range(self.N):
                velocidad_inicial[particulas] = self.situacion_inicial[f"particula_{particulas}"]['velocidad']
                Energias_iniciales[particulas] = 0.5 * self.m * (np.linalg.norm(velocidad_inicial[particulas]))**2
            Energias_iniciales_total = np.sum(Energias_iniciales) 
            print(f"Energía Cinética Total Inicial: {Energias_iniciales_total}")
            print(f"Energía Cinética Total Final: {Energia_total_final}")
        return None
    def comprobar_velocidades_finales(self):
        """
        Docstring for comprobar_velocidades_finales
        :param self: Description
        """
        velocidad_final = np.zeros((self.N,2))
        velocid_inicial = np.zeros((self.N,2))
        for particula in range(self.N):
            velocidad_final[particula] = self.velocidades_finales[f"particula_{particula}"]
            velocid_inicial[particula] = self.situacion_inicial[f"particula_{particula}"]['velocidad']
            
            print(velocidad_final == velocid_inicial)
            
        return None


## Desglose por partes

### Inciar de forma Random la posicion inicial i dirección de la velocidad

In [6]:
def inicializar_variables_necesarias(N, v_0, l_c):
    """
    Inicializa las variables necesarias para la simulacion:
    - Posiciones iniciales de las particulas (array de Nx2)
    - Velocidades iniciales de las particulas (array de Nx2)
    Argumentos:
    N = numero de particulas
    v_0 = velocidad inicial de las particulas (modulo)
    l_c = lado de la caja
    Retorna:
    variables_iniciales: dict con las variables iniciales
    """
    Particulas = {}
    for i in range(N):
        Particulas[f"particula_{i}"] = {
            'posicion': np.zeros(2),
            'velocidad': np.zeros(2)
        }
    # Inicializamos las posiciones
    posiciones = np.random.rand(N, 2) * l_c
    # Inicializamos las velocidades
    angulos = np.random.rand(N) * 2 * np.pi
    velocidades = np.zeros((N, 2))
    velocidades[:, 0] = v_0 * np.cos(angulos)
    velocidades[:, 1] = v_0 * np.sin(angulos)
    # Creamos el diccionario de variables iniciales
    for i in range(N):
        Particulas[f"particula_{i}"]['posicion'] = posiciones[i]
        Particulas[f"particula_{i}"]['velocidad'] = velocidades[i]
    return Particulas

print('='*100)
print('VARIABLES INICIALES PARA LA SIMULACION DE PARTICULAS EN 2D CONFINADAS EN UNA CAJA')
print('='*100)
print('\n')
print(inicializar_variables_necesarias(N, v_0, l_c))

VARIABLES INICIALES PARA LA SIMULACION DE PARTICULAS EN 2D CONFINADAS EN UNA CAJA


{'particula_0': {'posicion': array([0.49432194, 0.94581691]), 'velocidad': array([ 0.23092178, -0.97297232])}, 'particula_1': {'posicion': array([0.72101588, 0.71211633]), 'velocidad': array([ 0.97531002, -0.22084014])}, 'particula_2': {'posicion': array([0.56950812, 0.13270551]), 'velocidad': array([-0.95285447, -0.30342768])}, 'particula_3': {'posicion': array([0.26179319, 0.93006549]), 'velocidad': array([ 0.8690891 , -0.49465557])}, 'particula_4': {'posicion': array([0.21467796, 0.92138656]), 'velocidad': array([0.98239371, 0.18682237])}, 'particula_5': {'posicion': array([0.76047226, 0.48281249]), 'velocidad': array([ 0.88585577, -0.46396073])}, 'particula_6': {'posicion': array([0.59465753, 0.79633778]), 'velocidad': array([-0.27839241, -0.96046742])}, 'particula_7': {'posicion': array([0.21177352, 0.21022221]), 'velocidad': array([-0.97867872,  0.20539707])}, 'particula_8': {'posicion': array([0.